# Evaluate a Fine-Tuned LoRA Adapter

The eval notebook for **every** fine-tuned adapter — r=16 (Task #6) and the r=8 / r=32 rank ablations (Task #7). It loads the **base** Mistral-7B in 4-bit, layers the chosen LoRA adapter on top, runs `scripts/05_eval_finetuned.py` over the locked `data/eval/eval.jsonl`, and writes `results/runs/<RUN_NAME>/` — using the *same* ROUGE + Gemini-judge pipeline as the baseline, so every run is comparable.

**To pick which adapter:** set `RUN_NAME` in the **Run Parameter** cell (section 2.5) below, then run top-to-bottom.

**Runtime budget**: ~30–45 min on a free Colab T4 (50 generations @ 512 tokens + judge calls).

**Before you run this notebook, make sure:**
1. Runtime → Change runtime type → **T4 GPU** is selected.
2. The adapter you want to eval is on Drive at `/MyDrive/LLM_Finetune/models/adapters/<RUN_NAME>/`.
3. `HF_TOKEN` **and** `GEMINI_API_KEY` are set in Colab Secrets (🔑 left sidebar).
4. The latest commits are pushed to GitHub — Colab pulls a fresh copy below.

## 1. Confirm GPU is available

Should print a T4 with ~15 GB VRAM. If `nvidia-smi` fails or shows no GPU, switch the runtime type and restart.

In [ ]:
!nvidia-smi

## 2. Clone the project

Replace `<your-username>/LLM_Finetune` with your actual repo. For a private repo, use `https://<token>@github.com/<you>/LLM_Finetune.git` with a fine-grained PAT that has read access.

In [ ]:
!git clone https://github.com/Gabriel-Kevorkian/LLM_Finetune.git
%cd LLM_Finetune

## 2.5 Run parameter — **EDIT THIS FOR EACH RUN**

The only cell you change between runs. `RUN_NAME` selects both the adapter to load (`models/adapters/<RUN_NAME>`) and the output folder (`results/runs/<RUN_NAME>`). Every cell below interpolates `{RUN_NAME}` / `{ADAPTER}` — no other hardcoded paths.

In [ ]:
# === RUN PARAMETER — set which adapter to evaluate, then run top-to-bottom ===
#   r=16 (Task #6, done): "r16"   |   r=8 ablation: "r8"   |   r=32 ablation: "r32"
# RUN_NAME must match a trained adapter folder on Drive.
RUN_NAME = "r8"
ADAPTER  = f"models/adapters/{RUN_NAME}"
print(f"RUN_NAME = {RUN_NAME}")
print(f"ADAPTER  = {ADAPTER}")

## 3. Install dependencies

We need **`unsloth`** to load the 4-bit base the adapter was trained on, plus our `requirements.txt` for `rouge_score`, `google-genai`, `python-dotenv`, etc.

**Why we do NOT `pip install -U trl peft accelerate bitsandbytes`**: it breaks Unsloth's compiled cache (pinned to a specific TRL version). Let Unsloth pin its own deps. Install takes ~2 minutes.

In [ ]:
%%capture
# NO `-U` on trl/peft/accelerate/bitsandbytes — that would break Unsloth's
# compiled cache. unsloth's own setup.py is the source of truth for pins.
!pip install -q unsloth datasets
!pip install -q -r requirements.txt

## 4. Load secrets

- `HF_TOKEN` — required to download the gated base Mistral weights.
- `GEMINI_API_KEY` — required for the LLM-judge (`gemini-3.1-flash-lite`, free tier). `src/eval/llm_judge.py` reads it from the environment.

(No W&B here — evaluation isn't logged as a training run.)

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]               = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GEMINI_API_KEY"]         = userdata.get("GEMINI_API_KEY")

print(f"HF_TOKEN:        {'set' if os.environ.get('HF_TOKEN') else 'MISSING'}")
print(f"GEMINI_API_KEY:  {'set' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")

## 5. Mount Google Drive and redirect output dirs to Drive

**Two reasons:** (1) the adapter (gitignored, ~161 MB) lives only on Drive — symlinking `models/adapters/` makes `models/adapters/<RUN_NAME>` resolve to it; (2) symlinking `results/runs/` persists the eval output across runtime restarts. Same symlink trick as the training notebook. You'll get a Drive auth popup the first time.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, pathlib

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/LLM_Finetune")
(DRIVE_ROOT / "models/adapters").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "results/runs").mkdir(parents=True, exist_ok=True)

# Symlink local output dirs INTO Drive so reads find the adapter and writes
# (eval results) land on Drive. Always recreated, so re-running is safe.
for local_path_str, drive_path in [
    ("/content/LLM_Finetune/models/adapters", DRIVE_ROOT / "models/adapters"),
    ("/content/LLM_Finetune/results/runs",    DRIVE_ROOT / "results/runs"),
]:
    local_path = pathlib.Path(local_path_str)
    if local_path.is_symlink():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(drive_path, local_path)
    print(f"  {local_path} -> {drive_path}")

print("\nDrive mounted and output dirs symlinked.")

## 6. Verify the adapter is reachable

Sanity check before spending GPU time: confirm `models/adapters/<RUN_NAME>/` resolves (through the symlink) to the adapter on Drive and contains the LoRA weights.

In [ ]:
import os

print(f"adapter dir exists?  {os.path.isdir(ADAPTER)}")
if os.path.isdir(ADAPTER):
    !ls -lh {ADAPTER}
    assert os.path.exists(f"{ADAPTER}/adapter_model.safetensors"), \
        "adapter_model.safetensors missing — is the Drive path correct?"
    print("\nAdapter looks good — ready to eval.")
else:
    print(f"[STOP] Adapter not found at {ADAPTER}. Re-run section 5 and confirm the")
    print(f"       {RUN_NAME} adapter is at /MyDrive/LLM_Finetune/{ADAPTER}/")

## 7. Run the evaluation

Runs `scripts/05_eval_finetuned.py`: loads base + the `{RUN_NAME}` adapter (two-step PEFT load), generates an answer for each of the 50 eval questions (greedy, 512 tokens — same as baseline), scores with ROUGE-1/L + EM + Gemini judge, and writes `results/runs/{RUN_NAME}/{per_example.csv,results.json}` — symlinked to Drive.

**Runtime**: ~30–45 min on a T4. Don't close the tab.

In [ ]:
!python scripts/05_eval_finetuned.py --adapter {ADAPTER} --run-name {RUN_NAME}

## 8. Inspect the results

Print the headline ROUGE + judge numbers (next to baseline and r16 for context) and confirm both output files landed on Drive.

In [ ]:
import json

with open(f"results/runs/{RUN_NAME}/results.json") as f:
    summary = json.load(f)

agg = summary["aggregate"]
print(f"=== {RUN_NAME} fine-tuned — aggregate ===")
print(f"  rouge1:      {agg['rouge1']:.4f}   (baseline 0.132, r16 0.274)")
print(f"  rougeL:      {agg['rougeL']:.4f}   (baseline 0.107, r16 0.188)")
print(f"  exact_match: {agg['exact_match']:.4f}")
if agg.get("judge_score") is not None:
    print(f"  judge_score: {agg['judge_score']:.2f} / 5   (baseline 2.28, r16 3.06)")

print("\nFiles on Drive:")
!ls -lh results/runs/{RUN_NAME}/

## 9. Commit the results back to git

`results/runs/<RUN_NAME>/results.json` + `per_example.csv` are small and reproducible — they belong in git for the README audit trail. (The adapter stays gitignored.)

**You git-push from your local machine**: copy those two files from your Drive folder into the local repo, then `git add` + `git commit` + `git push`. Then run `python scripts/06_compile_results.py` locally to refresh the ablation table + chart, and update README §6.